In [51]:
!pip install --upgrade jax jaxlib

In [52]:
import os
# Force TensorFlow to use the CPU and ignore the CUDA error
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
# Upgrade libraries to resolve the JAX/TensorFlow attribute error
!pip install --upgrade jax jaxlib

In [ ]:
import pandas as pd
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

In [ ]:
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

## 2. LOAD DATA

In [ ]:
X_train = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/X_train.npy')
X_test = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/X_test.npy')
y_train = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/y_train.npy')
y_test = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/y_test.npy')

## 3. BUILDING LSTM MODEL

In [ ]:
# We use 16 units to reduce the parameter count for our 40 samples
def build_model():
    model = Sequential([
        Input(shape=(5, 5)),
        # Change this in your LSTM layer
        LSTM(16, activation='tanh', kernel_regularizer=regularizers.l2(0.05)),
        Dropout(0.2),
        Dense(8, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

## 4. TRAIN WITH BALANCED WEIGHTS

In [ ]:
model = build_model()
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
# Using more moderate class weights to avoid the 100% positive prediction bias
history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=4,
    validation_split=0.2,
    class_weight={0: 1.2, 1: 0.8}, 
    callbacks=[early_stop],
    verbose=0
)

## 5. BALANCED EVALUATION

In [ ]:
y_pred_prob = model.predict(X_test)
# Reset threshold to 0.5 to stop the model from over-predicting the positive class
y_pred = (y_pred_prob > 0.6).astype("int32")

## 6. RESULTS

In [67]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
print("\n" + "="*30)
print(f"STABILIZED ACCURACY: {accuracy:.4f}")
print(f"F1 SCORE: {f1:.4f}")
print("CONFUSION MATRIX:")
print(cm)
print("="*30)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step

STABILIZED ACCURACY: 0.8000
F1 SCORE: 0.8000
CONFUSION MATRIX:
[[4 2]
 [0 4]]


## 8. SAVE OUTPUTS

In [69]:
results_df.to_csv('lstm_final_results.csv', index=False)
model.save('lstm_financial_model.keras')
print("\n✓ LSTM Model and results saved.")


✓ LSTM Model and results saved.
